In [ ]:
#importing libraries
import os
import re
import sklearn
import pandas as pd
from tqdm.notebook import tqdm, trange
from matplotlib import pyplot as plt
import numpy
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score, auc, roc_curve, RocCurveDisplay
from sklearn.metrics import confusion_matrix,classification_report
import seaborn as sns
import tensorflow_hub as hub
from tensorflow.keras.layers import Input,Flatten, Lambda,LSTM, Bidirectional, Dense, Dropout, BatchNormalization, Conv2D,MaxPooling1D,Conv1D, GRU
from tensorflow.keras.models import Model
from tensorflow import keras
from tensorflow.keras import regularizers
import tensorflow as tf
from keras.layers import concatenate
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping 
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import class_weight


In [ ]:
#loading datasets
trainData=pd.read_csv("rota_virus_Fold_4_training_dataset_preproccessed.csv")
valData = pd.read_csv("rota_virus_Fold_4_validation_dataset_preproccessed.csv")
testData=pd.read_csv("rota_virus_test_dataset_preproccessed.csv")


In [ ]:
#encoding values.
trainDataset_3D = []
testDataset_3D = []
valDataset_3D = []

for i in trange(len(trainData)):
    sequence=[]
    for x in trainData['SEQUENCE'][i]:
        if x=='A':
            sequence.append([1,0,0,0,0])
        elif x=='C':
            sequence.append([0,1,0,0,0])
        elif x=='G':
            sequence.append([0,0,1,0,0])
        elif x=='T':
            sequence.append([0,0,0,1,0])
        elif x=='N':
            sequence.append([0,0,0,0,1])
    trainDataset_3D.append(sequence)
for i in trange(len(testData)):
    sequence=[]
    for x in testData['SEQUENCE'][i]:
        if x=='A':
            sequence.append([1,0,0,0,0])
        elif x=='C':
            sequence.append([0,1,0,0,0])
        elif x=='G':
            sequence.append([0,0,1,0,0])
        elif x=='T':
            sequence.append([0,0,0,1,0])
        elif x=='N':
            sequence.append([0,0,0,0,1])
    testDataset_3D.append(sequence)
for i in trange(len(valData)):
    sequence=[]
    for x in valData['SEQUENCE'][i]:
        if x=='A':
            sequence.append([1,0,0,0,0])
        elif x=='C':
            sequence.append([0,1,0,0,0])
        elif x=='G':
            sequence.append([0,0,1,0,0])
        elif x=='T':
            sequence.append([0,0,0,1,0])
        elif x=='N':
            sequence.append([0,0,0,0,1])
    valDataset_3D.append(sequence)

trainDataset_3D = numpy.array(trainDataset_3D)
testDataset_3D = numpy.array(testDataset_3D)
valDataset_3D = numpy.array(valDataset_3D)

In [ ]:
host_map = {
    
   
    'Homo sapiens'          :0,
    'Sus scrofa'            :1,
    'Bos taurus'            :2, 
    'Equus caballus'        :3,
    'Sus scrofa domesticus' :4,
    'Gallus gallus'         :5,
    'Canis lupus familiaris':6,
    'Capra hircus'          :7,
    'Vicugna pacos'         :8,
    'Bos grunniens'         :9,
    'Columba livia'         :10,
    'Aves'                  :11,
    0: 'Homo sapiens',
    1: 'Sus scrofa',
    2: 'Bos taurus',
    3: 'Equus caballus',
    4: 'Sus scrofa domesticus',
    5: 'Gallus gallus',
    6: 'Canis lupus familiaris',
    7: 'Capra hircus',
    8: 'Vicugna pacos',
    9: 'Bos grunniens',
    10: 'Columba livia',
    11: 'Aves',
    
}
maps = {
    'HOST' : host_map,
}

In [ ]:
#converting host labels to numerical value
trainData['HOST'] = trainData['HOST'].apply(lambda x : maps["HOST"][x])
testData['HOST'] = testData['HOST'].apply(lambda x : maps["HOST"][x])
valData['HOST'] = valData['HOST'].apply(lambda x : maps["HOST"][x])


In [ ]:
def build_model():
            input_layer = Input(shape=trainDataset_3D.shape[1:], name="input_layer")
            BiLSTM_Model = Bidirectional(keras.layers.LSTM(128, return_sequences= False), name="BiLSTM_Model")(input_layer)
            DROPOUT = Dropout(0.2, name="DROPOUT")(BiLSTM_Model)
            BatchNormalized = BatchNormalization(axis = -1, name="BatchNormalized")(DROPOUT)
            Dense_Layer1 = Dense(64, activation='relu', name="Dense_Layer1")(BatchNormalized)
            output = Dense(12, activation='softmax', name="output")(Dense_Layer1)                                                                                                                                                       
            model = Model(inputs=[input_layer], outputs=output, name="BILSTM")
            model.summary()
            model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
            return model

In [ ]:
fold_no = 5

#train the model for each of the fold
rota_model = build_model()
checkpoint = ModelCheckpoint('Rota_best_weight_Fold_'+str(fold_no)+'_Epoch-{epoch:03d}-valACC-{val_accuracy:.4f}.h5', 
        verbose=1, 
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        mode='max'
)

model_rota = rota_model.fit([trainDataset_3D], trainData['HOST'],validation_data=([valDataset_3D], valData['HOST']), epochs=200, batch_size=128,callbacks=[checkpoint])
rota_model.save_weights('BILSTM_Class_Balanced_Rota_Fold_'+str(fold_no)+'_Epoch1to200.h5')



In [ ]:
#loading the best weights
rota_model.load_weights('Rota_best_weight_Fold_4_Epoch-169-valACC-0.9674.h5')


In [ ]:
#testing the model

testCore = rota_model.evaluate(testDataset_3D, testData['HOST'], batch_size=128)
print(testCore)

In [ ]:
category_list = [ 'Homo sapiens',
    'Sus scrofa',
    'Bos taurus', 
    'Equus caballus',
    'Sus scrofa domesticus',
    'Gallus gallus',
    'Canis lupus familiaris',
    'Capra hircus',
    'Vicugna pacos',
    'Bos grunniens',
    'Columba livia',
    'Aves']

In [ ]:
predictions = rota_model.predict(testDataset_3D, batch_size=128, verbose=1)

In [ ]:
def get_prediction_labels(prediction_probabilities):
    return numpy.argmax(prediction_probabilities)

def get_labels():
    actual_label=[]
    for i in trange(testData['HOST'].shape[0]):
        actual_label.append(testData['HOST'][i])
    return actual_label


In [ ]:
predicted_labels = [get_prediction_labels(prediction) for prediction in tqdm(predictions)]
actual_labels = get_labels()

In [ ]:
print(classification_report(actual_labels, predicted_labels))

In [ ]:
#generating confusion matrix
cm = confusion_matrix(actual_labels, predicted_labels)
cmn = cm.astype('float')/ cm.sum(axis=1)[:, numpy.newaxis]
fig, ax = plt.subplots(figsize=(15,15))
sns.heatmap(cmn, annot=True, fmt='.4f',cmap="Oranges", xticklabels=category_list, yticklabels=category_list)
plt.ylabel('True Host')
plt.xlabel('Predicted Host');
plt.yticks(rotation=45) 
plt.xticks(rotation=45) 
plt.savefig('rota_confusion_matrix.png', dpi=1200, 
         format='png',
        )
plt.show(block=False)


In [ ]:
label_binarizer = LabelBinarizer().fit(trainData['HOST'])
y_onehot_test = label_binarizer.transform(testData['HOST'])
y_onehot_test.shape

In [ ]:
#multiclass roc curve
fpr = {} # False Positive Rate
tpr = {} # True Positive Rate
thresh ={} # Threshold
roc_auc = dict()

for i in range(12):
    fpr[i], tpr[i], thresh[i] = roc_curve(y_onehot_test[:, i], predictions[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    
    plt.plot(fpr[i], tpr[i], linestyle='--', 
             label='%s vs Rest (AUC=%0.2f)'%(category_list[i],roc_auc[i]))


plt.plot([0,1],[0,1],'b--')
plt.xlim([0,1])
plt.ylim([0,1.05])
plt.title('Multiclass ROC curve of Rota Virus')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive rate')
plt.legend(loc='lower right')
plt.savefig('rota_roc_curve.png', dpi=1200,
          format='png',
        )
plt.show()



In [ ]:
micro_roc_auc_ovr = roc_auc_score(
    y_onehot_test,
    predictions,
    multi_class="ovr",
    average="micro",
)

print(f"Micro-averaged One-vs-Rest ROC AUC score:\n{micro_roc_auc_ovr:.4f}")